# Notebook 3 — GAN Training for Synthetic Data Augmentation

**Project:** An Improved Computer Vision Model for Food Classification in Smart Refrigerators using GAN-Based Data Augmentation  
**Author:** Premshakthi Sekar | MSc Artificial Intelligence — Northumbria University  

---

## Purpose
This notebook trains a **Generative Adversarial Network (GAN)** to synthesise new refrigerator food images. The generated images are used to augment the original dataset, addressing class imbalance and increasing training data diversity for the improved classifier in Notebook 4.

## What is a GAN?
A GAN consists of two neural networks trained in competition:
- **Generator:** Takes random noise as input and generates synthetic images
- **Discriminator:** Tries to distinguish between real images and fake (generated) images

Through adversarial training, the Generator improves until it can produce images realistic enough to fool the Discriminator.

## Architecture Summary
- **Generator input:** 100-dimensional noise vector
- **Generator output:** 80×80×3 RGB synthetic image
- **Discriminator input:** 80×80×3 image (real or fake)
- **Training:** 10,000 epochs with batch size 64

> ⚠️ **Note:** Full GAN training took approximately 7 days on available hardware. Sample generated images are included in the `/sample_images/` folder to demonstrate output quality at various training stages.

## Step 1: Import Libraries

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported.')

## Step 2: Load & Preprocess Dataset for GAN Training

For GAN training, images are resized to 80×80 (smaller than the classifier) and normalised to the range [-1, 1]. This normalisation is standard for GAN training as it helps stabilise the generator's output.

In [ ]:
# Path to dataset — update to match your local path
dataset_path = 'dataset/gan_images'

images = []
labels = []

classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]

for class_name in classes:
    class_dir = os.path.join(dataset_path, class_name)
    for filename in os.listdir(class_dir):
        filepath = os.path.join(class_dir, filename)
        if filepath.endswith('.jpg') or filepath.endswith('.png'):
            img = Image.open(filepath).convert('RGB')
            img = img.resize((80, 80))  # GAN uses smaller image size
            images.append(np.array(img))
            labels.append(class_name)

# Encode labels
label_encoder = LabelEncoder()
numerical_labels = label_encoder.fit_transform(labels)

# Train/test split
x_train, x_test, y_train, y_test = train_test_split(
    np.array(images), numerical_labels,
    test_size=0.2, stratify=numerical_labels, random_state=42
)

# Normalise to [-1, 1] — standard for GAN training
x_train = (x_train.astype('float32') - 127.5) / 127.5

print(f'GAN training data shape: {x_train.shape}')
print(f'Pixel value range: [{x_train.min():.2f}, {x_train.max():.2f}]')

## Step 3: Build the Generator

The Generator transforms a 100-dimensional random noise vector into an 80×80 RGB image through a series of transposed convolution (upsampling) layers.

In [ ]:
def build_generator():
    """
    Generator Network:
    Input:  100-dimensional noise vector
    Output: 80x80x3 synthetic RGB image
    
    Architecture:
    Dense → Reshape → Conv2DTranspose (upsample) × 2 → Output
    """
    generator = models.Sequential([
        layers.Input(shape=(100,)),
        
        # Expand noise vector into spatial feature map
        layers.Dense(20 * 20 * 128, activation='relu'),
        layers.Reshape((20, 20, 128)),
        
        # Upsample: 20x20 → 40x40
        layers.Conv2DTranspose(128, (5, 5), strides=(2, 2), padding='same', activation='relu'),
        
        # Upsample: 40x40 → 80x80
        layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', activation='relu'),
        
        # Output: 80x80x3 RGB image, sigmoid to constrain values
        layers.Conv2DTranspose(3, (5, 5), padding='same', activation='sigmoid'),
    ], name='Generator')
    
    return generator

generator = build_generator()
generator.summary()

## Step 4: Build the Discriminator

The Discriminator is a binary classifier that predicts whether an input image is real or generated. LeakyReLU activations and Dropout are used to improve training stability.

In [ ]:
def build_discriminator():
    """
    Discriminator Network:
    Input:  80x80x3 image (real or generated)
    Output: Probability [0,1] — 1=real, 0=fake
    
    Architecture:
    Conv2D (downsample) × 2 → Flatten → Dense(1, sigmoid)
    LeakyReLU and Dropout used for training stability
    """
    discriminator = models.Sequential([
        layers.Input(shape=(80, 80, 3)),
        
        # Downsample: 80x80 → 40x40
        layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        
        # Downsample: 40x40 → 20x20
        layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        
        layers.Flatten(),
        
        # Binary output: real (1) or fake (0)
        layers.Dense(1, activation='sigmoid'),
    ], name='Discriminator')
    
    return discriminator

discriminator = build_discriminator()
discriminator.summary()

## Step 5: GAN Training Loop

The GAN is trained using an adversarial loop:
1. Generate fake images from noise
2. Train Discriminator on real and fake images
3. Train Generator to fool the Discriminator
4. Save sample generated images every 10 epochs to monitor progress

In [ ]:
def save_generated_images(epoch, generator, examples=16, figsize=(7, 7)):
    """Save a grid of generated images to monitor GAN training progress."""
    noise = np.random.normal(0, 1, [examples, 100])
    generated_images = generator.predict(noise, verbose=0)
    
    fig, axes = plt.subplots(4, 4, figsize=figsize)
    for i, ax in enumerate(axes.flat):
        ax.imshow(generated_images[i])
        ax.axis('off')
    plt.suptitle(f'GAN Generated Images — Epoch {epoch}', fontsize=12)
    plt.tight_layout()
    
    os.makedirs('gan_progress', exist_ok=True)
    plt.savefig(f'gan_progress/epoch_{epoch:05d}.png', dpi=100, bbox_inches='tight')
    plt.close()

print('save_generated_images function defined.')

In [ ]:
def gan_train(generator, discriminator, dataset, batch_size=64, epochs=10000):
    """
    Main GAN training loop.
    
    Each epoch:
    1. For each batch of real images:
       a. Generate fake images from noise
       b. Train discriminator on real images (label=1)
       c. Train discriminator on fake images (label=0)
       d. Train generator through combined GAN (label=1, to fool discriminator)
    2. Save sample images every 10 epochs
    
    Note: Full training took ~7 days on available hardware.
    """
    # Build combined GAN model
    gan = models.Sequential([generator, discriminator])
    
    # Compile discriminator
    discriminator.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    # Compile GAN (discriminator frozen during generator training)
    discriminator.trainable = False
    gan.compile(
        optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
        loss='binary_crossentropy'
    )
    discriminator.trainable = True
    
    for epoch in range(epochs):
        for real_images, _ in dataset:
            current_batch_size = len(real_images)
            
            # --- Train Discriminator ---
            noise = tf.random.normal([current_batch_size, 100])
            generated_images = generator.predict(noise, verbose=0)
            
            real_labels = tf.ones((current_batch_size, 1))
            fake_labels = tf.zeros((current_batch_size, 1))
            
            d_loss_real = discriminator.train_on_batch(real_images, real_labels)
            d_loss_fake = discriminator.train_on_batch(generated_images, fake_labels)
            
            # --- Train Generator ---
            noise = tf.random.normal([current_batch_size, 100])
            g_loss = gan.train_on_batch(noise, real_labels)
            
        # Log progress every 100 epochs
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1}/{epochs} | '
                  f'D_real_acc: {d_loss_real[1]:.3f} | '
                  f'D_fake_acc: {d_loss_fake[1]:.3f} | '
                  f'G_loss: {g_loss:.4f}')
        
        # Save generated images every 10 epochs
        if (epoch + 1) % 10 == 0:
            save_generated_images(epoch + 1, generator)

print('GAN training function defined.')
print('\nTo run training:')
print('  batch_size = 64')
print('  train_dataset = tf.data.Dataset.from_tensor_slices((x_train, y_train)).batch(batch_size)')
print('  gan_train(build_generator(), build_discriminator(), train_dataset, batch_size=64, epochs=10000)')
print('\n⚠️  Warning: Full training takes approximately 7 days on CPU.')

## Step 6: Visualise GAN Training Progression

The images below show how the GAN output evolved across training epochs. Early epochs produce noisy, unrecognisable images. As training progresses, the generated images become increasingly realistic.

In [ ]:
# Display sample dataset images (real refrigerator images used for training)
sample_dir = 'sample_images'

if os.path.exists(sample_dir):
    sample_files = [f for f in os.listdir(sample_dir) if f.endswith('.jpg') or f.endswith('.png')][:6]
    
    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    for i, (ax, fname) in enumerate(zip(axes.flat, sample_files)):
        img = Image.open(os.path.join(sample_dir, fname))
        ax.imshow(img)
        ax.set_title(f'Sample {i+1}', fontsize=10)
        ax.axis('off')
    
    plt.suptitle('Real Refrigerator Dataset Samples', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('real_dataset_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Sample images directory not found. Add images to sample_images/ folder.')

---
## Summary

| Component | Detail |
|-----------|--------|
| Generator input | 100-dimensional noise vector |
| Generator output | 80×80×3 RGB synthetic image |
| Discriminator | Binary classifier (real vs fake) |
| Training epochs | 10,000 |
| Batch size | 64 |
| Training time | ~7 days |
| Key challenge | Early-stage GAN produces noisy images (see sample_images/) |

**Observation:** While the GAN eventually produced recognisable food images, early stages generated noisy outputs with visible artefacts. This is expected behaviour in GAN training and is documented in the dissertation.

**Next:** Notebook 4 — Food Classification After GAN Integration (Improved Model)